### Societal Features Extraction

In [1]:
import numpy as np
import pandas as pd

#Change city
city = 'Texas'

#Data categories
Age_list = ['Under 5','5 to 17','18 to 24','25 to 44','45 to 54','55 to 64','65 to 74','75 and over']
Race_list = ['White','Black or African American','American Indian','Asian','Some other race','Two or more races']
Education_list = ['Less than high school graduate','High school graduate','College or associate','Bachelor','Graduate or professional']
Income_list = ['Less than 10k','10k to 15k','15k to 25k','25k to 35k','35k to 50k','50k to 75k','75k to 100k','100k to 150k','150k to 200k','200k or more']
Travel_list = ['Drive alone','Carpool','Public transportation','Walk','Bicycle','Other','Work at home']

#S0601: Age, Race, Education
S0601 = pd.read_csv(city + '_ZIP_S0601.csv', low_memory=False)
Age = S0601[list(map(lambda x: 'S0601_C01_0' + x + 'E', [str(i).rjust(2,'0') for i in range(2,10)]))]
Age.columns = Age_list
Race = S0601[list(map(lambda x: 'S0601_C01_0' + x + 'E', [str(i).rjust(2,'0') for i in range(14,21)]))]
Race.columns = Race_list[:4] + ['Native'] + Race_list[4:]
Education = S0601[list(map(lambda x: 'S0601_C01_0' + x + 'E', [str(i).rjust(2,'0') for i in range(33,38)]))]
Education.columns = Education_list
#S1901: Income
S1901 = pd.read_csv(city + '_ZIP_S1901.csv', low_memory=False)
Income = S1901[list(map(lambda x: 'S1901_C01_0' + x + 'E', [str(i).rjust(2,'0') for i in range(2,12)]))]
Income.columns = Income_list
#S0801: Travel
S0801 = pd.read_csv(city + '_ZIP_S0801.csv', low_memory=False)
Travel = S0801[list(map(lambda x: 'S0801_C01_0' + x + 'E', [str(i).rjust(2,'0') for i in [3,4] + list(range(9,14))]))]
Travel.columns = Travel_list

#Data aggregation and cleaning
df = pd.concat([S0601['NAME'], Age, Race, Education, Income, Travel], axis=1)
df = df.drop(index=[0,1])
df = df.replace('-', np.nan)
df = df.dropna()
df.iloc[:,1:] = df.iloc[:,1:].applymap(lambda x: 0.01 * float(x))
df['Some other race'] = df['Some other race'] + df['Native']

#Data output
with pd.ExcelWriter(city + '_Features.xlsx') as writer:
    pd.concat([df['NAME'],df[Age_list]], axis=1).to_excel(writer,'Sheet0',index=False)
    pd.concat([df['NAME'],df[Race_list]], axis=1).to_excel(writer,'Sheet1',index=False)
    pd.concat([df['NAME'],df[Education_list]], axis=1).to_excel(writer,'Sheet2',index=False)
    pd.concat([df['NAME'],df[Income_list]], axis=1).to_excel(writer,'Sheet3',index=False)
    pd.concat([df['NAME'],df[Travel_list]], axis=1).to_excel(writer,'Sheet4',index=False)

### SDI Dataset

In [2]:
import math
import pandas as pd

#Change city
city = 'Texas'

SDI = pd.read_excel(city + '_Features.xlsx','Sheet0')[['NAME']]
Features = ['Age','Race','Education','Income','Travel']

for i in range(len(Features)):
    df = pd.read_excel(city + '_Features.xlsx','Sheet' + str(i))
    df.iloc[:,1:] = df.iloc[:,1:].apply(lambda x: x / x.sum(), axis=1)
    df.iloc[:,1:] = df.iloc[:,1:].applymap(lambda x: -x * math.log(x) if x != 0.0 else 0.0)
    SDI[Features[i]] = df.iloc[:,1:].apply(lambda x: x.sum() / math.log(df.shape[1]-1), axis=1)

#Data output
SDI.iloc[:,1:] = SDI.iloc[:,1:].round(4)
SDI.to_csv('SDI_Dataset.csv',index=False)

### Entropy Weight Method

In [3]:
import math
import pandas as pd

#Change city
city = 'Texas'

df = pd.read_csv('SDI_Dataset.csv')
row = df.shape[0]
col = df.shape[1]
df.iloc[:,1:] = df.iloc[:,1:].apply(lambda x: x / max(x), axis=1)
df.iloc[:,1:] = df.iloc[:,1:].apply(lambda x: x / x.sum())
df.iloc[:,1:] = df.iloc[:,1:].applymap(lambda x: -x * math.log(x) if x != 0.0 else 0.0)

E_list = df.iloc[:,1:].apply(lambda x: x.sum() / math.log(row)).tolist()
df['Diversity'] = 0
for i in range(len(E_list)):
    W = (1 - E_list[i]) / (col - 1 - sum(E_list))
    df['Diversity'] += W * pd.read_csv('SDI_Dataset.csv').iloc[:,i+1]

#Data output
df['Diversity'] = df['Diversity'].round(4)
df[['NAME','Diversity']].to_csv(city + '_ZIP_Diversity.csv',index=False)